# Deepfake Image Detector — Google Drive → Colab → Android TFLite

This notebook trains a real-vs-fake **image** classifier from your `Real vs Fake Faces` dataset, evaluates it, compares Keras and TFLite outputs, and exports `deepfake_detection_model.tflite` for your Android app.

**Important:** No notebook can guarantee the best result on every image. Performance depends on dataset quality, label correctness, duplicate leakage, and whether test images resemble images users upload. This notebook uses a held-out test set, transfer learning, fine-tuning, class weights, and evaluation metrics.

**Android contract for this notebook**
- Input: `[1, 224, 224, 3]`, `float32`, RGB pixels `0..255`
- Output: one sigmoid value = `P(REAL)`
- Labels: `Fake = 0`, `Real = 1`
- Android mapping: `realProbability = output[0]`, `fakeProbability = 1 - output[0]`

EfficientNetB0's Keras implementation includes its own input rescaling. Do not divide the input by 255 in Android for this exported model.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set the dataset path

First, upload the `Real vs Fake Faces` folder to Google Drive (or upload a ZIP to Drive and extract it there). Then set `DATASET_DIR` below to the **folder containing your dataset**.

Example:
`/content/drive/MyDrive/Real vs Fake Faces`

Supported layouts:
- `Real vs Fake Faces/Real/...` and `Real vs Fake Faces/Fake/...`
- `Real vs Fake Faces/train/Real/...`, `train/Fake/...`, and optional `val/` and `test/` folders.

Folder names should be recognizable as real/fake. If your folder names differ, edit the aliases in the next cell.

In [ ]:
from pathlib import Path
import os, random, json, shutil, math
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, balanced_accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_DIR = "/content/drive/MyDrive/Real vs Fake Faces"  # EDIT THIS
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
assert os.path.isdir(DATASET_DIR), f"Folder not found: {DATASET_DIR}. Edit DATASET_DIR."

## 3. Find Real/Fake folders and make train/validation/test splits

In [ ]:
FAKE_NAMES = {"fake", "fakes", "forged", "synthetic", "ai", "ai_generated", "deepfake", "0"}
REAL_NAMES = {"real", "reals", "genuine", "authentic", "original", "1"}
SPLITS = {
    "train": {"train", "training"},
    "val": {"val", "valid", "validation", "dev"},
    "test": {"test", "testing"}
}

def norm_name(name):
    return name.strip().lower().replace(" ", "_").replace("-", "_")

def class_dirs(folder):
    folder = Path(folder)
    found = {}
    if not folder.is_dir():
        return found
    for p in folder.iterdir():
        if p.is_dir():
            n = norm_name(p.name)
            if n in FAKE_NAMES:
                found[0] = p
            elif n in REAL_NAMES:
                found[1] = p
    return found

def images_in(folder, label):
    return [(str(p), int(label)) for p in Path(folder).rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS]

root = Path(DATASET_DIR)
split_dirs = {}
for p in root.iterdir():
    if p.is_dir():
        n = norm_name(p.name)
        for key, names in SPLITS.items():
            if n in names:
                split_dirs[key] = p
                break

train_records, val_records, test_records = [], [], []
has_split_train = "train" in split_dirs and bool(class_dirs(split_dirs["train"]))

if has_split_train:
    for label, folder in class_dirs(split_dirs["train"]).items():
        train_records += images_in(folder, label)
    for key, target in [("val", val_records), ("test", test_records)]:
        if key in split_dirs:
            for label, folder in class_dirs(split_dirs[key]).items():
                target += images_in(folder, label)

    # If no validation folder exists, reserve part of training data.
    if not val_records:
        labels = [y for _, y in train_records]
        train_records, val_records = train_test_split(
            train_records, test_size=0.15, random_state=SEED, stratify=labels
        )
    # If no test folder exists, reserve a test set from the remaining training records.
    if not test_records:
        labels = [y for _, y in train_records]
        train_records, test_records = train_test_split(
            train_records, test_size=0.15, random_state=SEED, stratify=labels
        )
else:
    roots = class_dirs(root)
    if roots:
        all_records = []
        for label, folder in roots.items():
            all_records += images_in(folder, label)
    else:
        # Handle an extra wrapper folder, e.g. root/Real vs Fake Faces/Real and /Fake.
        candidates = [p for p in root.iterdir() if p.is_dir()]
        found = [(p, class_dirs(p)) for p in candidates if class_dirs(p)]
        if len(found) != 1:
            raise ValueError(
                "Could not locate Fake/Real class folders. Inspect the dataset folder and "
                "set DATASET_DIR to the directory directly containing Fake and Real."
            )
        nested_root, roots = found[0]
        all_records = []
        for label, folder in roots.items():
            all_records += images_in(folder, label)

    if not all_records:
        raise ValueError("No supported images found in Fake/Real folders.")
    labels = [y for _, y in all_records]
    train_records, temp = train_test_split(
        all_records, test_size=0.30, random_state=SEED, stratify=labels
    )
    temp_labels = [y for _, y in temp]
    val_records, test_records = train_test_split(
        temp, test_size=0.50, random_state=SEED, stratify=temp_labels
    )

def show_counts(name, records):
    c = Counter(y for _, y in records)
    print(f"{name}: {len(records):,} images | Fake={c.get(0,0):,} | Real={c.get(1,0):,}")

show_counts("Train", train_records)
show_counts("Validation", val_records)
show_counts("Test", test_records)
assert all([train_records, val_records, test_records]), "One of the splits is empty."
assert set(y for _, y in train_records) == {0,1}, "Training split must contain Fake and Real."

## 4. Create image pipelines and modest augmentation

In [ ]:
def load_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], antialias=True)
    # Float RGB pixel values 0..255. EfficientNetB0 performs its own rescaling.
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.float32)

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.08),
], name="training_augmentation")

def make_ds(records, training=False):
    paths = [p for p, _ in records]
    labels = [y for _, y in records]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(records), 12000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x,y: (augment(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(train_records, True)
val_ds = make_ds(val_records)
test_ds = make_ds(test_records)

# Compute class weights to reduce bias when class counts are imbalanced.
counts = Counter(y for _, y in train_records)
total = sum(counts.values())
class_weight = {label: total / (2.0 * count) for label, count in counts.items()}
print("Class weights:", class_weight)

images, labels = next(iter(train_ds))
plt.figure(figsize=(9, 7))
for i in range(min(9, len(images))):
    plt.subplot(3,3,i+1)
    plt.imshow(tf.cast(tf.clip_by_value(images[i],0,255), tf.uint8).numpy())
    plt.title("Real" if int(labels[i].numpy()) == 1 else "Fake")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Build EfficientNetB0 transfer-learning model

In [ ]:
base = keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False

inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="rgb_0_255")
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.25)(x)
real_score = layers.Dense(1, activation="sigmoid", name="real_score")(x)
model = keras.Model(inputs, real_score, name="deepfake_image_detector")

metrics = [
    keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
    keras.metrics.Precision(name="precision", thresholds=0.5),
    keras.metrics.Recall(name="recall", thresholds=0.5),
    keras.metrics.AUC(name="auc")
]
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=metrics
)
model.summary()

In [ ]:
OUT_DIR = "/content/deepfake_model_output"
os.makedirs(OUT_DIR, exist_ok=True)
best_path = os.path.join(OUT_DIR, "best_deepfake_model.keras")
callbacks = [
    keras.callbacks.ModelCheckpoint(best_path, monitor="val_auc", mode="max",
                                    save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                                      patience=2, min_lr=1e-6, verbose=1)
]
history1 = model.fit(
    train_ds, validation_data=val_ds, epochs=12,
    class_weight=class_weight, callbacks=callbacks
)

## 6. Fine-tune last layers

In [ ]:
# Unfreeze the later EfficientNet layers for careful fine-tuning.
base.trainable = True
for layer in base.layers[:-35]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
        keras.metrics.Precision(name="precision", thresholds=0.5),
        keras.metrics.Recall(name="recall", thresholds=0.5),
        keras.metrics.AUC(name="auc")
    ]
)
history2 = model.fit(
    train_ds, validation_data=val_ds, epochs=8,
    class_weight=class_weight, callbacks=callbacks
)

## 7. Evaluate on held-out test set

In [ ]:
# Load the checkpoint with the best validation AUC, not necessarily the last epoch.
model = keras.models.load_model(best_path)

print("Held-out test metrics:")
print(model.evaluate(test_ds, return_dict=True))

y_true = np.concatenate([y.numpy().astype(int) for _, y in test_ds])
y_score = model.predict(test_ds, verbose=0).reshape(-1)
y_pred = (y_score >= 0.5).astype(int)

print("\nClassification report (Fake=0, Real=1):")
print(classification_report(y_true, y_pred, labels=[0,1],
                            target_names=["Fake","Real"], digits=4, zero_division=0))
print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
if len(np.unique(y_true)) == 2:
    print("ROC AUC:", round(roc_auc_score(y_true, y_score), 4))

cm = confusion_matrix(y_true, y_pred, labels=[0,1])
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Held-out test confusion matrix")
plt.xticks([0,1], ["Pred Fake","Pred Real"])
plt.yticks([0,1], ["Actual Fake","Actual Real"])
for (i,j), v in np.ndenumerate(cm):
    plt.text(j,i,str(v),ha="center",va="center")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 8. Choose a threshold using validation data (optional, do not tune on test set)

In [ ]:
# Threshold can trade off missed fakes vs false alarms. This chooses the threshold
# that maximizes balanced accuracy on validation data only.
val_y = np.concatenate([y.numpy().astype(int) for _, y in val_ds])
val_scores = model.predict(val_ds, verbose=0).reshape(-1)
thresholds = np.linspace(0.10, 0.90, 81)
threshold_results = []
for t in thresholds:
    pred = (val_scores >= t).astype(int)
    threshold_results.append((balanced_accuracy_score(val_y, pred), float(t)))
best_bal_acc, chosen_threshold = max(threshold_results)
print(f"Validation-selected threshold for REAL score: {chosen_threshold:.2f}")
print(f"Validation balanced accuracy at threshold: {best_bal_acc:.4f}")
print("Use this threshold in the app only if you also store/use it consistently.")

## 9. Export float32 TFLite model and verify output

In [ ]:
tflite_path = os.path.join(OUT_DIR, "deepfake_detection_model.tflite")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = []  # float32 first, to match Android FLOAT32 preprocessing
tflite_bytes = converter.convert()
with open(tflite_path, "wb") as f:
    f.write(tflite_bytes)
print("TFLite saved:", tflite_path)
print("Size MB:", round(os.path.getsize(tflite_path)/(1024*1024), 2))

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print("Input:", inp)
print("Output:", out)
assert inp["shape"].tolist() == [1,224,224,3]
assert inp["dtype"] == np.float32

In [ ]:
# Compare Keras and TFLite scores on the same held-out test image.
# Use the first test image path:
sample_path, sample_label = test_records[0]
pil = Image.open(sample_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample = np.asarray(pil, dtype=np.float32)[None, ...]

keras_real = float(model.predict(sample, verbose=0).reshape(-1)[0])
interpreter.set_tensor(inp["index"], sample)
interpreter.invoke()
tflite_real = float(interpreter.get_tensor(out["index"]).reshape(-1)[0])

print("Ground truth:", "REAL" if sample_label == 1 else "FAKE")
print(f"Keras P(REAL):  {keras_real:.6f} ({keras_real*100:.2f}%)")
print(f"TFLite P(REAL): {tflite_real:.6f} ({tflite_real*100:.2f}%)")
print(f"Difference: {abs(keras_real-tflite_real)*100:.4f} percentage points")
plt.imshow(pil); plt.axis("off"); plt.show()

## 10. Download model and optional Keras checkpoint

In [ ]:
from google.colab import files
files.download(tflite_path)
# Optional: download the Keras checkpoint for future retraining.
# files.download(best_path)

## 11. Install in Android Studio

1. Back up the existing file.
2. Replace `app/src/main/assets/deepfake_detection_model.tflite` with the downloaded file (keep the exact filename).
3. Keep the Android input as RGB `FLOAT32` values `0..255` for this notebook's model.
4. For this model, output tensor has one sigmoid score for REAL. Android should use that score as `realProbability`; `fakeProbability = 1 - realProbability`.
5. Rebuild and reinstall the app. Confirm Android Logcat's raw output is close to the TFLite score printed above for the same image.

### Accuracy note
If performance is weak, check class-folder labels, remove duplicate/near-duplicate images across splits, add more varied real and fake examples, and evaluate on an independent source. Avoid tuning on the test set. A detector should be allowed to say "Inconclusive" when confidence is low; a high score is not proof of authenticity.